---

##SPRINT 11

---



In [ ]:
# Configuración inicial y librerías para SPRINT 11
# ================================================

# Librerías para manipulación, carga y análisis de datos
import pandas as pd
import numpy as np

# Herramientas del entorno (Google Colab)
from google.colab import drive

# Configuraciones globales de visualización para Pandas
pd.options.display.float_format = '{:.2f}'.format

# Conexión al entorno
drive.mount('/content/drive')

In [ ]:
# 1. CARGA Y PREPARACIÓN DE COMPAÑÍAS
# ===================================
ruta_companias = "/content/drive/MyDrive/Colab Notebooks/sprint9/N1-Ex.8__companies.csv"
companias = pd.read_csv(ruta_companias, sep = ",")
companias = companias.rename(columns={"country": "country_company"})

# 2. CARGA Y PREPARACIÓN DE PRODUCTOS
# ===================================
ruta_productos = "/content/drive/MyDrive/Colab Notebooks/sprint9/N1-Ex.8__products.csv"
productos = pd.read_csv(ruta_productos, sep=",")
productos = productos.rename(columns={'id': 'product_id'})

# 3. CARGA, CONCATENACIÓN Y LIMPIEZA DE USUARIOS
# ===================================================
ruta_usuarios_eu = "/content/drive/MyDrive/Colab Notebooks/sprint9/N1-Ex.8__european_users.csv"
ruta_usuario_usa = "/content/drive/MyDrive/Colab Notebooks/sprint9/N1-Ex.8__american_users.csv"

usuarios_eu = pd.read_csv(ruta_usuarios_eu, sep=",")
usuarios_usa = pd.read_csv(ruta_usuario_usa, sep =",")

usuarios_eu['continent_user'] = 'Europa'
usuarios_usa['continent_user'] = 'América'

# Concatenar ambos dataframes
usuarios = pd.concat([usuarios_usa, usuarios_eu], ignore_index=True)
usuarios = usuarios.sort_values(by='id', ascending=True).reset_index(drop=True)

usuarios = usuarios.rename(columns={
  'id': 'user_id',
  'country': 'country_user',
  'city': 'city_user',
  'address': 'address_user',
  'phone': 'phone_user',
  'email': 'email_user'
})

# Transformaciones específicas de Usuarios
# Asegurar formato fecha en birth_date
usuarios['birth_date'] = pd.to_datetime(usuarios['birth_date'])

# Nivel 1 - Punto 6: Combinar nombre y apellido
usuarios['full_name'] = usuarios['name'].fillna('') + ' ' + usuarios['surname'].fillna('')

# Nivel 1 - Punto 6: Calcular edad basándonos en el año del análisis (2026)
# Nivel 1 - Punto 6: Calcular edad basándonos en el año actual dinámico
usuarios['age'] = pd.Timestamp.now().year - usuarios['birth_date'].dt.year

# 4. CARGA DE TRANSACCIONES Y CREACIÓN DE TABLA PUENTE
# ===================================================
# Agrega aquí la ruta correcta de tu archivo de transacciones original
ruta_transacciones = "/content/drive/MyDrive/Colab Notebooks/sprint9/N1-Ex.8__transactions.csv"
transacciones = pd.read_csv(ruta_transacciones, sep=";")

# Transformaciones de Transacciones y Tabla Puente
# Asegurar formato fecha en timestamp
transacciones['timestamp'] = pd.to_datetime(transacciones['timestamp'])

# Construcción de la tabla puente para el Nivel 3
trx_products = transacciones[['id', 'product_ids']].copy()
trx_products = trx_products.rename(columns={'id': 'trx_id'})

# Separar y "explotar" la columna de productos agrupados
trx_products['product_ids'] = trx_products['product_ids'].fillna('')
trx_products['product_id'] = trx_products['product_ids'].str.split(',')
trx_products = trx_products.explode('product_id')

# Limpieza de espacios en blanco y registros vacíos
trx_products['product_id'] = trx_products['product_id'].str.strip()
trx_products = trx_products.drop(columns=['product_ids'])
trx_products = trx_products[trx_products['product_id'] != '']

# 5. EXPORTACIÓN A FORMATO PARQUET
# ===================================================
# Guardamos en la misma carpeta para que los descargues juntos
OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/sprint9/"

transacciones.to_parquet(OUTPUT_PATH + 'transacciones_clean.parquet', index=False)
usuarios.to_parquet(OUTPUT_PATH + 'usuarios_clean.parquet', index=False)
trx_products.to_parquet(OUTPUT_PATH + 'trx_products_clean.parquet', index=False)
companias.to_parquet(OUTPUT_PATH + 'companias_clean.parquet', index=False)
productos.to_parquet(OUTPUT_PATH + 'productos_clean.parquet', index=False)

print("¡Todo listo! Los 5 archivos Parquet se han exportado correctamente y guardaron sus tipos de datos.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
¡Todo listo! Los 5 archivos Parquet se han exportado correctamente y guardaron sus tipos de datos.
